In [ ]:
import sys

import pandas as pd

from from_root import from_root

sys.path.insert(0, str(from_root("src")))

from read_and_write_docs import read_jsonl, read_rds
from model_loading import load_model
from n_gram_tracing import tokenize_to_tokens, common_ngrams

In [ ]:
known = read_jsonl("/Volumes/BCross/datasets/author_verification/test/Enron Cleaned/known_raw.jsonl")
unknown = read_jsonl("/Volumes/BCross/datasets/author_verification/test/Enron Cleaned/unknown_raw.jsonl")

metadata = read_rds("/Volumes/BCross/datasets/author_verification/test/metadata.rds")

tokenizer = load_model("/Volumes/BCross/models/gpt2", load_model=False)

In [76]:
prob_num = 0

corpus_metadata = metadata[metadata['corpus'] == 'Enron Cleaned'].copy()

selected_problem = corpus_metadata.iloc[prob_num]

known_author = selected_problem['known_author']
unknown_author = selected_problem['unknown_author']
problem_name = selected_problem['problem']

known_text_list = known[known['author'] == known_author]['text'].tolist()
unknown_text = unknown[unknown['author'] == unknown_author]['text'].iloc[0]

corpus_metadata.iloc[[prob_num]]

,problem,corpus,known_author,unknown_author
12434,Kevin.hyatt vs Kevin.hyatt,Enron Cleaned,Kevin.hyatt,Kevin.hyatt


In [77]:
actual_results = read_rds("/Volumes/BCross/av_datasets_experiments/weighted_ngram_tracing_concat/test/combined_scores_raw.rds")

In [79]:
actual_results[actual_results['corpus'] == 'Enron Cleaned']

,data_type,corpus,scoring_model,problem,known_author,unknown_author,target,num_problems,ngrams_found,completed,weight,min_token_size,simpson_score,jaccard_score,alpha,base
33351,test,Enron Cleaned,gpt2,Kevin.hyatt vs Kevin.hyatt,Kevin.hyatt,Kevin.hyatt,True,4.0,4.0,True,linear,1.0,0.022709,0.004844,NaN,NaN
33352,test,Enron Cleaned,gpt2,Kevin.hyatt vs Kevin.hyatt,Kevin.hyatt,Kevin.hyatt,True,4.0,4.0,True,linear,2.0,0.014519,0.003075,NaN,NaN
33353,test,Enron Cleaned,gpt2,Kevin.hyatt vs Kevin.hyatt,Kevin.hyatt,Kevin.hyatt,True,4.0,4.0,True,linear,3.0,0.004973,0.001045,NaN,NaN
33354,test,Enron Cleaned,gpt2,Kevin.hyatt vs Kevin.hyatt,Kevin.hyatt,Kevin.hyatt,True,4.0,4.0,True,linear,4.0,0.002540,0.000532,NaN,NaN
33355,test,Enron Cleaned,gpt2,Kevin.hyatt vs Kevin.hyatt,Kevin.hyatt,Kevin.hyatt,True,4.0,4.0,True,linear,6.0,0.001166,0.000244,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
34894,test,Enron Cleaned,gpt2,W.white vs W.white,W.white,W.white,True,3.0,3.0,True,exp,3.0,0.003390,0.000888,NaN,2.0
34895,test,Enron Cleaned,gpt2,W.white vs W.white,W.white,W.white,True,3.0,3.0,True,exp,4.0,0.002356,0.000616,NaN,2.0
34896,test,Enron Cleaned,gpt2,W.white vs W.white,W.white,W.white,True,3.0,3.0,True,exp,5.0,0.002077,0.000543,NaN,2.0
34897,test,Enron Cleaned,gpt2,W.white vs W.white,W.white,W.white,True,3.0,3.0,True,exp,6.0,0.001731,0.000452,NaN,2.0


## Get tokens

In [39]:
unknown_tokens = tokenize_to_tokens(unknown_text, tokenizer=tokenizer, lowercase=True)

known_tokens_list = []

for known_text in known_text_list:
    
    known_tokens = tokenize_to_tokens(known_text, tokenizer=tokenizer, lowercase=True)
    
    known_tokens_list.append(known_tokens)

## Get Common N-Grams and Lengths

In [56]:
common_list = []

for known_text in known_text_list:
    common = common_ngrams(known_text, unknown_text, tokenizer=tokenizer, lowercase=True)
    common_list.extend(common)

distinct_common_ngrams = list(dict.fromkeys(tuple(x) for x in common_list))
distinct_common_ngrams = [list(x) for x in distinct_common_ngrams]

distinct_common_ngrams = sorted(
    distinct_common_ngrams,
    key=lambda x: (len(x), sum(len(str(token)) for token in x))
)

distinct_lengths = sorted({len(x) for x in distinct_common_ngrams})

## Generate token sets

In [41]:
def overlapping_windows(items, n):
    """
    Return overlapping sublists of length n.

    Example:
        overlapping_windows(["a", "b", "c"], 2)
        -> [["a", "b"], ["b", "c"]]
    """
    if n <= 0:
        raise ValueError("n must be greater than 0")
    
    return [items[i:i+n] for i in range(len(items) - n + 1)]

In [63]:
unknown_token_list_len = []
known_token_list_len = []

for length in distinct_lengths:
    
    # Get the unknown tokens for each length
    unknown_tokens_n = overlapping_windows(unknown_tokens, length)
    
    # Get the distinct unknown tokens for each length
    distinct_unknown_tokens_n = list(dict.fromkeys(tuple(x) for x in unknown_tokens_n))
    distinct_unknown_tokens_n = [list(x) for x in distinct_unknown_tokens_n]
    
    # Now we need to loop across the known tokens
    known_tokens_temp = []
    
    # Get the known tokens for each text and append
    for known_tokens in known_tokens_list:
        
        known_tokens_n = overlapping_windows(known_tokens, length)
        known_tokens_temp.extend(known_tokens_n)
        
    distinct_known_tokens_n = list(dict.fromkeys(tuple(x) for x in known_tokens_temp))
    distinct_known_tokens_n = [list(x) for x in distinct_known_tokens_n]
    
    print(f"Len - {length} | Unknown Tokens - {len(unknown_tokens_n)} | Known Tokens - {len(known_tokens_temp)}")
    print(f"Len - {length} | Distinct Unknown Tokens - {len(distinct_unknown_tokens_n)} | Distinct Known Tokens - {len(distinct_known_tokens_n)}")
    
    unknown_token_list_len.append(distinct_unknown_tokens_n)
    known_token_list_len.append(distinct_known_tokens_n)

Len - 1 | Unknown Tokens - 863 | Known Tokens - 3261
Len - 1 | Distinct Unknown Tokens - 446 | Distinct Known Tokens - 1174
Len - 2 | Unknown Tokens - 862 | Known Tokens - 3257
Len - 2 | Distinct Unknown Tokens - 786 | Distinct Known Tokens - 2693
Len - 3 | Unknown Tokens - 861 | Known Tokens - 3253
Len - 3 | Distinct Unknown Tokens - 842 | Distinct Known Tokens - 3135
Len - 4 | Unknown Tokens - 860 | Known Tokens - 3249
Len - 4 | Distinct Unknown Tokens - 855 | Distinct Known Tokens - 3214
Len - 6 | Unknown Tokens - 858 | Known Tokens - 3241
Len - 6 | Distinct Unknown Tokens - 858 | Distinct Known Tokens - 3240
Len - 7 | Unknown Tokens - 857 | Known Tokens - 3237
Len - 7 | Distinct Unknown Tokens - 857 | Distinct Known Tokens - 3237


In [80]:
def compute_overlap_scores(
    token_lengths,
    distinct_common_ngrams,
    distinct_unknown_tokens_n,
    distinct_known_tokens_n
):
    """
    Compute Simpson-style and Jaccard overlap scores by token length.

    Parameters
    ----------
    token_lengths : list[int]
        Token lengths to evaluate.
    distinct_common_ngrams : list[list]
        Flat list of distinct shared n-grams across all lengths.
    distinct_unknown_tokens_n : list[list[list]]
        Grouped distinct unknown n-grams, aligned with token_lengths.
    distinct_known_tokens_n : list[list[list]]
        Grouped distinct known n-grams, aligned with token_lengths.

    Returns
    -------
    list[dict]
        One dict per token length with counts and scores.
    """
    if not (
        len(token_lengths) == len(distinct_unknown_tokens_n) == len(distinct_known_tokens_n)
    ):
        raise ValueError(
            "token_lengths, distinct_unknown_tokens_n, and distinct_known_tokens_n "
            "must have the same length"
        )

    results = []

    for n, unknown_ngrams, known_ngrams in zip(
        token_lengths, distinct_unknown_tokens_n, distinct_known_tokens_n
    ):
        common_ngrams_n = [x for x in distinct_common_ngrams if len(x) == n]

        common_count = len(common_ngrams_n)
        unknown_count = len(unknown_ngrams)
        known_count = len(known_ngrams)

        simpson_score = common_count / unknown_count if unknown_count > 0 else 0.0

        union_count = unknown_count + known_count - common_count
        jaccard_score = common_count / union_count if union_count > 0 else 0.0

        results.append({
            "token_length": n,
            "common_count": common_count,
            "unknown_count": unknown_count,
            "known_count": known_count,
            "union_count": union_count,
            "simpson_score": simpson_score,
            "jaccard_score": jaccard_score
        })

    return results

In [72]:
len(distinct_common_ngrams)


280

In [81]:
results = compute_overlap_scores(
    distinct_lengths,
    distinct_common_ngrams,
    unknown_token_list_len,
    known_token_list_len
)

In [85]:
results_df = pd.DataFrame(results)
results_df

,token_length,common_count,unknown_count,known_count,union_count,simpson_score,jaccard_score
0,1,163,446,1174,1457,0.365471,0.111874
1,2,93,786,2693,3386,0.118321,0.027466
2,3,16,842,3135,3961,0.019002,0.004039
3,4,6,855,3214,4063,0.007018,0.001477
4,6,1,858,3240,4097,0.001166,0.000244
5,7,1,857,3237,4093,0.001167,0.000244


In [89]:
import pandas as pd

def rough_weighted_score(df: pd.DataFrame, score_col: str) -> float:
    """
    Compute a rough token-length-weighted average of a score column.
    """
    weights = df["token_length"]
    scores = df[score_col]

    weight_sum = weights.sum()
    if weight_sum == 0:
        return 0.0

    return (scores * weights).sum() / weight_sum

def aggregate_ge_with_rough_scores(results: pd.DataFrame, t: int) -> dict:
    sub = results[results["token_length"] >= t].copy()

    # Proper weighted counts
    num_w = (sub["common_count"] * sub["token_length"]).sum()
    den_simpson_w = (sub["unknown_count"] * sub["token_length"]).sum()
    known_w = (sub["known_count"] * sub["token_length"]).sum()
    den_jaccard_w = den_simpson_w + known_w - num_w

    simpson_score = num_w / den_simpson_w if den_simpson_w > 0 else 0.0
    jaccard_score = num_w / den_jaccard_w if den_jaccard_w > 0 else 0.0

    # Rough weighted averages of existing per-length scores
    weight_sum = sub["token_length"].sum()
    rough_simpson_score = (
        (sub["simpson_score"] * sub["token_length"]).sum() / weight_sum
        if weight_sum > 0 else 0.0
    )
    rough_jaccard_score = (
        (sub["jaccard_score"] * sub["token_length"]).sum() / weight_sum
        if weight_sum > 0 else 0.0
    )

    return {
        "min_token_size": t,
        "num_w": num_w,
        "den_simpson_w": den_simpson_w,
        "known_w": known_w,
        "den_jaccard_w": den_jaccard_w,
        "simpson_score": simpson_score,
        "jaccard_score": jaccard_score,
        "rough_simpson_score": rough_simpson_score,
        "rough_jaccard_score": rough_jaccard_score,
    }

def aggregate_results_df_weighted(results: pd.DataFrame) -> pd.DataFrame:
    thresholds = sorted(results["token_length"].drop_duplicates())
    rows = [aggregate_ge_with_rough_scores(results, t) for t in thresholds]
    return pd.DataFrame(rows)

In [90]:
aggregate_results_df_weighted(results_df)

,min_token_size,num_w,den_simpson_w,known_w,den_jaccard_w,simpson_score,jaccard_score,rough_simpson_score,rough_jaccard_score
0,1,434,19111,70920,89597,0.022709,0.004844,0.030537,0.008174
1,2,271,18665,69746,88140,0.014519,0.003075,0.015313,0.003461
2,3,85,17093,64360,81368,0.004973,0.001045,0.005012,0.001060
3,4,37,14567,54955,69485,0.002540,0.000532,0.002543,0.000534
4,6,13,11147,42099,53233,0.001166,0.000244,0.001166,0.000244
5,7,7,5999,22659,28651,0.001167,0.000244,0.001167,0.000244
